# 双编码器交叉注意力架构 - 宣传技术检测
## Configuration 3: Dual-Encoder with Cross-Attention

这个notebook实现了论文中表现最好的Config 3架构，包含：
- 双编码器（文本编码器 + 解释编码器）
- 双向交叉注意力机制
- 4种特征融合
- XLM-RoBERTa-base
- 模型保存到S3

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory of this repository
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 Task 3 development data
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirectories

# CheckThat! 2024 Task 3 data
CHECKTHAT_DIR = "your/path/here"

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


## 1. 环境配置和导入库

In [ ]:
# Standard libraries
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import tempfile
import boto3

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, RandomSampler

# Transformers
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

# PyTorch Lightning
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

# scikit-learn
from sklearn.metrics import f1_score, precision_score, recall_score

# Environment optimisation
os.environ["TOKENIZERS_PARALLELISM"] = 'false'
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# Set random seed
torch.manual_seed(42)

print("✅ Libraries imported")

## 2. 参数配置

In [ ]:
# GPU configuration
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Check CUDA availability
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ 使用设备: {device} - {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
else:
    device = torch.device('cpu')
    print(f"⚠️ CUDA不可用，使用CPU")

# Model hyperparameters
MODEL_NAME = 'xlm-roberta-base'  # 约125M参数
MAX_LENGTH = 256                  # 文本最大长度
MAX_EXPLANATION_LENGTH = 128      # 解释最大长度

# Training hyperparameters
EPOCHS = 10
BATCH_SIZE = 8
ACCUMULATE_GRAD_BATCHES = 4      # 有效batch size = 8*4=32
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.001
WARMUP_STEPS = 1000

# Loss function configuration
LOSS_TYPE = 'bce'  # 'bce' 或 'focal'
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 1.0

# Class-weight parameters
NEG_POS_RATIO = 3.0
MAX_WEIGHT = 30.0

# S3 configuration (credentials via env vars)
ENDPOINT = "https://your-s3-endpoint"  # Set to your S3-compatible endpoint
S3_BUCKET = "your-s3-bucket"
S3_PREFIX = "multi_label_models"
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "your-access-key-id")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")

# Data paths
TRAIN_FOLDER = 'your/path/here'  # Update to your local path
TRAIN_LABELS = 'your/path/here'  # Update to your local path
DEV_FOLDER = 'your/path/here'  # Update to your local path
DEV_LABELS = 'your/path/here'  # Update to your local path
EXPLANATIONS_FILE = 'your/path/here'  # Update to your local path
TECHNIQUES_FILE = 'your/path/here'  # Update to your local path

print(f"""
📋 配置摘要:
  模型: {MODEL_NAME}
  训练轮数: {EPOCHS}
  批次大小: {BATCH_SIZE}
  有效批次大小: {BATCH_SIZE * ACCUMULATE_GRAD_BATCHES}
  学习率: {LEARNING_RATE}
  损失函数: {LOSS_TYPE}
  文本长度: {MAX_LENGTH}
  解释长度: {MAX_EXPLANATION_LENGTH}
""")

## 3. S3存储功能

In [ ]:
def upload_to_s3(local_file_path, bucket_name, s3_key):
    """
    上传文件到S3
    
    Returns:
        (bool, str): (是否成功, S3 URI)
    """
    try:
        s3 = boto3.resource(
            's3',
            endpoint_url=ENDPOINT,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY
        )
        
        bucket = s3.Bucket(bucket_name)
        print(f"📤 正在上传到 {ENDPOINT}/{bucket_name}/{s3_key}...")
        bucket.upload_file(local_file_path, s3_key)
        
        s3_uri = f"{ENDPOINT}/{bucket_name}/{s3_key}"
        print(f"✅ 上传成功: {s3_uri}")
        return True, s3_uri
    except Exception as e:
        print(f"❌ 上传失败: {str(e)}")
        return False, None


def download_from_s3(bucket_name, s3_key, local_file_path):
    """
    从S3下载文件
    
    Returns:
        bool: 是否成功
    """
    try:
        s3 = boto3.resource(
            's3',
            endpoint_url=ENDPOINT,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY
        )
        
        os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
        print(f"📥 正在从 {ENDPOINT}/{bucket_name}/{s3_key} 下载...")
        s3.Bucket(bucket_name).download_file(s3_key, local_file_path)
        print(f"✅ 下载成功")
        return True
    except Exception as e:
        print(f"❌ 下载失败: {str(e)}")
        return False


class S3ModelCheckpoint(pl.Callback):
    """自动保存最佳模型到S3"""
    
    def __init__(self, bucket_name, s3_prefix, model_name, monitor='val_f1_micro', mode='max'):
        super().__init__()
        self.bucket_name = bucket_name
        self.s3_prefix = s3_prefix
        self.model_name = model_name
        self.monitor = monitor
        self.mode = mode
        self.best_model_score = None
        self.best_model_path = None
        self.temp_dir = tempfile.mkdtemp()
        self.compare = lambda x, y: x > y if self.mode == 'max' else x < y

    def on_validation_end(self, trainer, pl_module):
        current_score = trainer.callback_metrics.get(self.monitor)
        if current_score is None:
            return
        
        if isinstance(current_score, torch.Tensor):
            current_score = current_score.item()
            
        if self.best_model_score is None or self.compare(current_score, self.best_model_score):
            self.best_model_score = current_score
            
            filename = f"{self.model_name.split('/')[-1]}_{EPOCHS}_{LOSS_TYPE}_dual_encoder.ckpt"
            temp_path = os.path.join(self.temp_dir, filename)
            trainer.save_checkpoint(temp_path)
            
            s3_key = f"{self.s3_prefix}/{filename}"
            success, s3_uri = upload_to_s3(temp_path, self.bucket_name, s3_key)
            
            if success:
                self.best_model_path = s3_uri
                print(f"🏆 新的最佳模型 ({self.monitor}: {current_score:.4f})")
                os.remove(temp_path)
    
    def on_train_end(self, trainer, pl_module):
        import shutil
        try:
            shutil.rmtree(self.temp_dir)
        except:
            pass

print("✅ S3功能定义完成")

## 4. 数据处理函数

In [ ]:
def make_dataframe(data_type='train'):
    """
    创建数据框架
    
    Args:
        data_type: 'train' 或 'dev'
    
    Returns:
        DataFrame: 包含id, line, text, labels列
    """
    if data_type == 'train':
        input_folder = TRAIN_FOLDER
        labels_fn = TRAIN_LABELS
    elif data_type == 'dev':
        input_folder = DEV_FOLDER
        labels_fn = DEV_LABELS
    else:
        raise ValueError("data_type必须是'train'或'dev'")
    
    # Read article text files
    text = []
    skipped_files = 0
    for fil in tqdm(filter(lambda x: x.endswith('.txt'), os.listdir(input_folder))):
        iD = fil.split('.')[0]
        try:
            with open(input_folder + fil, 'r', encoding='utf-8') as f:
                content = f.read()
            lines = list(enumerate(content.splitlines(), 1))
            text.extend([(iD,) + line for line in lines])
        except UnicodeDecodeError:
            skipped_files += 1
            continue
    
    if skipped_files > 0:
        print(f"⚠️ 跳过了 {skipped_files} 个无法解码的文件")

    df_text = pd.DataFrame(text, columns=['id', 'line', 'text'])
    df_text.line = df_text.line.apply(int)
    df_text = df_text[df_text.text.str.strip().str.len() > 0].copy()
    df_text = df_text.set_index(['id', 'line'])

    # Read label file
    try:
        labels_data = []
        with open(labels_fn, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split('\t')
                if len(parts) < 2:
                    parts = line.split()
                if len(parts) >= 2:
                    id_val = parts[0]
                    try:
                        line_val = int(parts[1])
                        label_val = parts[2] if len(parts) >= 3 else ''
                        labels_data.append([id_val, line_val, label_val])
                    except ValueError:
                        continue
        
        if labels_data:
            labels = pd.DataFrame(labels_data, columns=['id', 'line', 'labels'])
            labels['line'] = labels['line'].astype(int)
            labels = labels.set_index(['id', 'line'])
            df = df_text.join(labels, how='left')[['text', 'labels']]
            df['labels'] = df['labels'].fillna('')
        else:
            raise ValueError("标签文件为空")
    except Exception as e:
        print(f"⚠️ 处理标签文件出错: {str(e)}，创建空标签")
        df = df_text.copy()
        df['labels'] = ''
        df = df[['text', 'labels']]
    
    # Clean text
    df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
    return df.reset_index()


def load_label_classes():
    """
    加载标签类别列表
    
    Returns:
        list: 排序后的标签名称列表
    """
    if not os.path.exists(TECHNIQUES_FILE):
        print(f"⚠️ 找不到 {TECHNIQUES_FILE}，使用默认标签")
        return sorted([
            "Appeal_to_Authority", "Appeal_to_Fear", "Appeal_to_Prejudice",
            "Bandwagon", "Black_and_White_Fallacy", "Causal_Oversimplification",
            "Doubt", "Exaggeration", "Flag_Waving", "Loaded_Language",
            "Name_Calling", "Reductio_ad_Hitlerum", "Repetition", "Slogans",
            "Straw_Man", "Thought_Terminating_Cliches", "Whataboutism"
        ])
    
    labels_name = []
    with open(TECHNIQUES_FILE, "r") as f:
        for line in f:
            labels_name.append(line.rstrip())
    return sorted(labels_name)


def load_explanations_data(explanations_file):
    """
    加载LLM生成的解释数据
    
    Args:
        explanations_file: TSV格式的解释文件路径
    
    Returns:
        dict: {(id, text): explanation}
    """
    explanations = {}
    try:
        df = pd.read_csv(explanations_file, sep='\t')
        for _, row in df.iterrows():
            key = (row['id'], row['text'])
            explanations[key] = row['analysis']
        print(f"✅ 加载了 {len(explanations)} 条解释数据")
    except Exception as e:
        print(f"❌ 加载解释数据失败: {str(e)}")
    return explanations


def load_multi_label_data_with_explanations(data_type='train', explanations=None):
    """
    加载多标签数据并整合解释
    
    Returns:
        tuple: (idxs, lines, texts, labels_tensor, explanation_texts, label_names)
    """
    df = make_dataframe(data_type=data_type)
    
    # Filter overly long texts
    max_text_len = 1000
    df = df[df['text'].str.len() <= max_text_len].copy()
    
    all_idxs = df["id"].to_numpy()
    all_lines = df["line"].to_numpy()
    all_data = [str(text) for text in df["text"].to_numpy()]
    
    # Load labels
    labels_name = load_label_classes()
    num_labels = len(labels_name)
    
    # Build multi-label matrix
    multi_labels = []
    for label_str in df['labels'].fillna('').values:
        label_vec = np.zeros(num_labels, dtype=np.float32)
        if label_str:
            labels = label_str.split(',')
            for label in labels:
                if label in labels_name:
                    label_idx = labels_name.index(label)
                    label_vec[label_idx] = 1.0
        multi_labels.append(label_vec)
    
    # Attach explanations
    explanation_texts = []
    if explanations:
        for idx, text in zip(all_idxs, all_data):
            key = (idx, text)
            explanation_texts.append(str(explanations.get(key, "")))
    else:
        explanation_texts = [""] * len(all_data)
    
    return (
        np.array(all_idxs, dtype=object),
        np.array(all_lines, dtype=np.int32),
        all_data,
        torch.tensor(np.array(multi_labels)),
        explanation_texts,
        labels_name
    )

print("✅ 数据处理Functions defined")

## 5. 数据集类

In [ ]:
class MultiLabelDatasetWithExplanations(torch.utils.data.Dataset):
    """
    双编码器数据集：分别处理文本和解释
    """
    
    def __init__(self, tokenizer, max_length, max_explanation_length, 
                 data_tuple, data_type='train'):
        self.max_length = max_length
        self.max_explanation_length = max_explanation_length
        self.idxs, self.lines, self.texts, self.y, self.explanations, self.label_names = data_tuple
        
        # Ensure texts are strings
        self.texts = [str(text) for text in self.texts]
        
        # Text encoder
        print(f"📝 正在编码{data_type}文本...")
        self.text_input_ids = []
        self.text_attention_mask = []
        
        batch_size = 256
        for i in tqdm(range(0, len(self.texts), batch_size)):
            batch = self.texts[i:i+batch_size]
            tokenized = tokenizer(
                batch,
                padding="max_length",
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt"
            )
            self.text_input_ids.append(tokenized["input_ids"])
            self.text_attention_mask.append(tokenized["attention_mask"])
        
        self.text_input_ids = torch.cat(self.text_input_ids, dim=0)
        self.text_attention_mask = torch.cat(self.text_attention_mask, dim=0)
        
        # Explanation encoder
        print(f"💡 正在编码{data_type}解释...")
        processed_explanations = [exp if exp and exp.strip() else "无解释" 
                                 for exp in self.explanations]
        
        self.exp_input_ids = []
        self.exp_attention_mask = []
        
        for i in tqdm(range(0, len(processed_explanations), batch_size)):
            batch = processed_explanations[i:i+batch_size]
            tokenized = tokenizer(
                batch,
                padding="max_length",
                truncation=True,
                max_length=self.max_explanation_length,
                return_tensors="pt"
            )
            self.exp_input_ids.append(tokenized["input_ids"])
            self.exp_attention_mask.append(tokenized["attention_mask"])
        
        self.exp_input_ids = torch.cat(self.exp_input_ids, dim=0)
        self.exp_attention_mask = torch.cat(self.exp_attention_mask, dim=0)
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        print(f"✅ {data_type}数据集准备完成: {len(self)} 个样本")

    def __getitem__(self, index):
        return (
            self.text_input_ids[index],
            self.text_attention_mask[index],
            self.exp_input_ids[index],
            self.exp_attention_mask[index],
            torch.squeeze(self.y[index]),
            self.idxs[index],
            torch.tensor(self.lines[index], dtype=torch.int64)
        )

    def __len__(self):
        return self.text_input_ids.shape[0]

print("✅ 数据集类定义完成")

## 6. 损失函数

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss实现
    用于处理类别不平衡问题
    """
    
    def __init__(self, alpha=1, gamma=2, pos_weight=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        epsilon = 1e-7
        targets_stable = torch.clamp(targets.float(), min=epsilon, max=1-epsilon)
        
        if self.pos_weight is not None:
            BCE_loss = F.binary_cross_entropy_with_logits(
                inputs, targets_stable, pos_weight=self.pos_weight, reduction='none'
            )
        else:
            BCE_loss = F.binary_cross_entropy_with_logits(
                inputs, targets_stable, reduction='none'
            )
        
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss


def compute_multi_label_class_weights(labels, neg_pos_ratio=3.0, max_weight=30.0):
    """
    计算多标签类别权重
    
    Args:
        labels: [samples, classes]的二值标签张量
        neg_pos_ratio: 负正样本比例系数
        max_weight: 权重上限
    
    Returns:
        torch.Tensor: 每个类别的权重
    """
    pos_counts = labels.sum(dim=0)
    total_samples = labels.shape[0]
    neg_counts = total_samples - pos_counts
    
    weights = []
    for i in range(labels.shape[1]):
        if pos_counts[i] > 0:
            weight = min((neg_counts[i] / pos_counts[i]) * neg_pos_ratio, max_weight)
            weights.append(weight)
        else:
            weights.append(1.0)
    
    return torch.tensor(weights, dtype=torch.float32, device=labels.device)

print("✅ 损失Functions defined")

## 7. 双编码器模型（核心）

In [ ]:
class DualEncoderClassifier(pl.LightningModule):
    """
    双编码器交叉注意力架构 (Configuration 3)
    
    特点:
    - 两个独立的XLM-RoBERTa编码器
    - 双向交叉注意力机制
    - 4种特征融合
    """
    
    def __init__(self, plm, num_labels, class_weights=None, 
                 learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS,
                 loss_type='bce', focal_gamma=2.0, focal_alpha=1.0):
        super().__init__()
        
        # 🔑 关键: 两个独立编码器
        self.text_encoder = AutoModel.from_pretrained(plm)
        self.explanation_encoder = AutoModel.from_pretrained(plm)
        
        # 启用梯度检查点
        if hasattr(self.text_encoder, "gradient_checkpointing_enable"):
            self.text_encoder.gradient_checkpointing_enable()
            self.explanation_encoder.gradient_checkpointing_enable()
        
        self.hidden_size = self.text_encoder.config.hidden_size
        self.num_labels = num_labels
        self.learning_rate = learning_rate
        self.warmup_steps = warmup_steps
        self.loss_type = loss_type
        
        # 🔑 关键: 双向交叉注意力
        self.text_to_exp_attention = nn.MultiheadAttention(
            embed_dim=self.hidden_size,
            num_heads=8,
            batch_first=True
        )
        
        self.exp_to_text_attention = nn.MultiheadAttention(
            embed_dim=self.hidden_size,
            num_heads=8,
            batch_first=True
        )
        
        # 🔑 关键: 特征融合层 (4个特征 -> 1个特征)
        self.fusion_layer = nn.Sequential(
            nn.Linear(self.hidden_size * 4, self.hidden_size * 2),
            nn.LayerNorm(self.hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        
        # Classification head
        self.classifier = nn.Linear(self.hidden_size, num_labels)
        
        # Loss function
        if loss_type == 'focal':
            self.criterion = FocalLoss(
                alpha=focal_alpha,
                gamma=focal_gamma,
                pos_weight=class_weights
            )
        else:
            self.criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
        
        # Metrics collection
        self.train_preds = []
        self.train_labels = []
        self.val_preds = []
        self.val_labels = []
    
    def forward(self, text_ids, text_mask, exp_ids=None, exp_mask=None):
        """
        前向传播
        
        如果没有解释，只使用文本编码器
        如果有解释，使用完整的双编码器+交叉注意力架构
        """
        # Text encoder
        text_outputs = self.text_encoder(
            input_ids=text_ids,
            attention_mask=text_mask,
            return_dict=True
        )
        text_hidden = text_outputs.last_hidden_state  # [batch, text_len, hidden]
        text_cls = text_hidden[:, 0, :]  # [batch, hidden]
        
        # No explanation: use text encoder only
        if exp_ids is None or exp_mask is None:
            return self.classifier(text_cls)
        
        # Explanation encoder
        exp_outputs = self.explanation_encoder(
            input_ids=exp_ids,
            attention_mask=exp_mask,
            return_dict=True
        )
        exp_hidden = exp_outputs.last_hidden_state  # [batch, exp_len, hidden]
        exp_cls = exp_hidden[:, 0, :]  # [batch, hidden]
        
        # 🔑 关键: 双向交叉注意力
        # Text attends to explanation
        text_to_exp_out, _ = self.text_to_exp_attention(
            query=text_hidden,
            key=exp_hidden,
            value=exp_hidden,
            key_padding_mask=~exp_mask.bool()
        )
        
        # Explanation attends to text
        exp_to_text_out, _ = self.exp_to_text_attention(
            query=exp_hidden,
            key=text_hidden,
            value=text_hidden,
            key_padding_mask=~text_mask.bool()
        )
        
        text_to_exp_cls = text_to_exp_out[:, 0, :]
        exp_to_text_cls = exp_to_text_out[:, 0, :]
        
        # 🔑 关键: 4种特征融合
        combined_feature = torch.cat([
            text_cls,          # 1. 原始文本表示
            exp_cls,           # 2. 原始解释表示
            text_to_exp_cls,   # 3. 文本关注解释
            exp_to_text_cls    # 4. 解释关注文本
        ], dim=1)  # [batch, hidden*4]
        
        # Fused representation
        fused_feature = self.fusion_layer(combined_feature)  # [batch, hidden]
        
        # Classification
        logits = self.classifier(fused_feature)
        return logits

    def training_step(self, batch, batch_idx):
        text_ids, text_mask, exp_ids, exp_mask, labels, _, _ = batch
        preds = self(text_ids, text_mask, exp_ids, exp_mask)
        
        # Ensure shape compatibility
        if labels.shape != preds.shape:
            labels = labels.view(*preds.shape)
        
        # Numerical stabilisation
        epsilon = 1e-7
        labels_stable = torch.clamp(labels.float(), min=epsilon, max=1-epsilon)
        
        loss = self.criterion(preds, labels_stable)
        
        if torch.isnan(loss):
            print(f"⚠️ 批次 {batch_idx} 出现NaN损失")
            return {"loss": torch.tensor(1e-5, requires_grad=True, device=loss.device)}
        
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        
        # Collect predictions
        with torch.no_grad():
            pred_probs = torch.sigmoid(preds)
            valid_mask = ~torch.isnan(pred_probs)
            if valid_mask.all():
                self.train_preds.extend(pred_probs.detach().cpu().numpy())
                self.train_labels.extend(labels.detach().cpu().numpy())
        
        # Periodically clear GPU cache
        if batch_idx % 50 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        text_ids, text_mask, exp_ids, exp_mask, labels, _, _ = batch
        
        # 检查输入
        if torch.isnan(text_ids).any() or torch.isnan(labels).any():
            print(f"⚠️ 批次 {batch_idx} 输入包含NaN")
            return {"val_loss": torch.tensor(0.0, device=self.device)}
        
        preds = self(text_ids, text_mask, exp_ids, exp_mask)
        
        if labels.shape != preds.shape:
            labels = labels.view(*preds.shape)
        
        epsilon = 1e-7
        labels_stable = torch.clamp(labels.float(), min=epsilon, max=1-epsilon)
        loss = self.criterion(preds, labels_stable)
        
        if torch.isnan(loss):
            print(f"⚠️ 批次 {batch_idx} 验证损失为NaN")
            return {"val_loss": torch.tensor(0.0, device=self.device)}
        
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        
        # 多标签评估
        with torch.no_grad():
            pred_probs = torch.sigmoid(preds)
            pred_labels = (pred_probs > 0.5).float()
            
            # 精确匹配
            exact_match = (pred_labels == labels).all(dim=1).float().mean()
            self.log('val_exact_match', exact_match, prog_bar=True)
            
            valid_mask = ~torch.isnan(pred_probs)
            if valid_mask.all():
                self.val_preds.extend(pred_probs.detach().cpu().numpy())
                self.val_labels.extend(labels.detach().cpu().numpy())
        
        return {"val_loss": loss}

    def on_train_epoch_end(self):
        """计算训练集指标"""
        if not self.train_preds:
            return
        
        pred_labels = (np.array(self.train_preds) > 0.5).astype(int)
        true_labels = np.array(self.train_labels)
        
        # Micro and macro averages
        metrics = {
            'train_precision_micro': precision_score(true_labels, pred_labels, average='micro', zero_division=0),
            'train_recall_micro': recall_score(true_labels, pred_labels, average='micro', zero_division=0),
            'train_f1_micro': f1_score(true_labels, pred_labels, average='micro', zero_division=0),
            'train_precision_macro': precision_score(true_labels, pred_labels, average='macro', zero_division=0),
            'train_recall_macro': recall_score(true_labels, pred_labels, average='macro', zero_division=0),
            'train_f1_macro': f1_score(true_labels, pred_labels, average='macro', zero_division=0)
        }
        
        for k, v in metrics.items():
            self.log(k, v)
        
        self.train_preds = []
        self.train_labels = []

    def on_validation_epoch_end(self):
        """计算验证集指标"""
        if not self.val_preds:
            return
        
        pred_labels = (np.array(self.val_preds) > 0.5).astype(int)
        true_labels = np.array(self.val_labels)
        
        metrics = {
            'val_precision_micro': precision_score(true_labels, pred_labels, average='micro', zero_division=0),
            'val_recall_micro': recall_score(true_labels, pred_labels, average='micro', zero_division=0),
            'val_f1_micro': f1_score(true_labels, pred_labels, average='micro', zero_division=0),
            'val_precision_macro': precision_score(true_labels, pred_labels, average='macro', zero_division=0),
            'val_recall_macro': recall_score(true_labels, pred_labels, average='macro', zero_division=0),
            'val_f1_macro': f1_score(true_labels, pred_labels, average='macro', zero_division=0)
        }
        
        for k, v in metrics.items():
            self.log(k, v, prog_bar=('f1' in k))
        
        # Per-label F1
        for i in range(self.num_labels):
            label_f1 = f1_score(true_labels[:, i], pred_labels[:, i], zero_division=0)
            if label_f1 > 0:
                self.log(f'val_f1_label_{i}', label_f1)
        
        self.val_preds = []
        self.val_labels = []

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.learning_rate,
            weight_decay=WEIGHT_DECAY
        )
        
        total_steps = self.trainer.estimated_stepping_batches
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=self.warmup_steps,
            num_training_steps=total_steps
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
                "frequency": 1,
                "monitor": "val_f1_micro",
            },
        }

print("✅ 双编码器模型定义完成")

## 8. 训练函数

In [ ]:
def train_dual_encoder_model():
    """
    训练双编码器模型的主函数
    """
    print("\n" + "="*50)
    print("🚀 开始训练双编码器交叉注意力模型")
    print("="*50 + "\n")
    
    # 1. 加载标签
    print("📋 步骤 1/8: 加载标签类别...")
    labels_name = load_label_classes()
    num_labels = len(labels_name)
    print(f"   找到 {num_labels} 个标签类别")
    
    # 2. 加载解释
    print("\n💡 步骤 2/8: 加载解释数据...")
    explanations = load_explanations_data(EXPLANATIONS_FILE)
    
    # 3. 加载分词器
    print("\n🔤 步骤 3/8: 加载分词器...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    print(f"   使用分词器: {MODEL_NAME}")
    
    # 4. 加载训练数据
    print("\n📚 步骤 4/8: 加载训练数据...")
    train_data = load_multi_label_data_with_explanations('train', explanations=explanations)
    idxs, lines, X, y, explanations_train, _ = train_data
    print(f"   训练样本数: {len(X)}")
    
    # 5. 加载验证数据
    print("\n📚 步骤 5/8: 加载验证数据...")
    dev_data = load_multi_label_data_with_explanations('dev', explanations=explanations)
    dev_idxs, dev_lines, dev_X, dev_y, explanations_dev, _ = dev_data
    print(f"   验证样本数: {len(dev_X)}")
    
    # 6. 计算类别权重
    print("\n⚖️  步骤 6/8: 计算类别权重...")
    y_train = y.numpy()
    pos_counts = y_train.sum(axis=0)
    total_samples = y_train.shape[0]
    
    print("   前10个标签的正样本统计:")
    for i, label in enumerate(labels_name[:10]):
        if pos_counts[i] > 0:
            print(f"   - {label}: {int(pos_counts[i])} ({pos_counts[i]/total_samples*100:.2f}%)")
    
    class_weights = compute_multi_label_class_weights(
        y, 
        neg_pos_ratio=NEG_POS_RATIO, 
        max_weight=MAX_WEIGHT
    )
    class_weights = class_weights.to(device)
    print(f"   权重范围: {class_weights.min():.2f} - {class_weights.max():.2f}")
    
    # 7. 创建数据集和加载器
    print("\n📦 步骤 7/8: 创建数据集和加载器...")
    dataset_train = MultiLabelDatasetWithExplanations(
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
        max_explanation_length=MAX_EXPLANATION_LENGTH,
        data_tuple=train_data,
        data_type='train'
    )
    
    dataset_val = MultiLabelDatasetWithExplanations(
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
        max_explanation_length=MAX_EXPLANATION_LENGTH,
        data_tuple=dev_data,
        data_type='dev'
    )
    
    train_sampler = RandomSampler(dataset_train)
    train_loader = DataLoader(
        dataset_train,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        num_workers=2,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        dataset_val,
        batch_size=BATCH_SIZE * 2,
        num_workers=2,
        pin_memory=True
    )
    
    print(f"   训练批次数: {len(train_loader)}")
    print(f"   验证批次数: {len(val_loader)}")
    
    # 8. 创建模型和训练器
    print("\n🏗️  步骤 8/8: 创建模型和训练器...")
    model = DualEncoderClassifier(
        plm=MODEL_NAME,
        num_labels=num_labels,
        class_weights=class_weights,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        loss_type=LOSS_TYPE,
        focal_gamma=FOCAL_GAMMA,
        focal_alpha=FOCAL_ALPHA
    )
    
    # Callbacks
    s3_checkpoint = S3ModelCheckpoint(
        bucket_name=S3_BUCKET,
        s3_prefix=f"{S3_PREFIX}/dual_encoder",
        model_name=MODEL_NAME,
        monitor='val_f1_micro',
        mode='max'
    )
    
    early_stop = EarlyStopping(
        monitor='val_f1_micro',
        patience=3,
        verbose=True,
        mode='max'
    )
    
    # Logger
    temp_log_dir = tempfile.mkdtemp(prefix="lightning_logs_")
    try:
        logger = TensorBoardLogger(
            save_dir=temp_log_dir,
            name='dual_encoder_model'
        )
    except:
        from pytorch_lightning.loggers import CSVLogger
        logger = CSVLogger(
            save_dir=temp_log_dir,
            name='dual_encoder_model'
        )
    
    # Trainer
    trainer = pl.Trainer(
        max_epochs=EPOCHS,
        callbacks=[s3_checkpoint, early_stop],
        precision="32",
        accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,
        gradient_clip_val=0.5,
        accelerator="auto",
        devices=1,
        enable_progress_bar=True,
        enable_model_summary=True,
        log_every_n_steps=10,
        logger=logger
    )
    
    # Start training
    print("\n" + "="*50)
    print("🏃 开始训练...")
    print("="*50 + "\n")
    
    try:
        model = model.to(device)
        trainer.fit(model, train_loader, val_loader)
        
        # Save results
        best_model_path = s3_checkpoint.best_model_path
        best_score = s3_checkpoint.best_model_score
        
        if best_model_path:
            print("\n" + "="*50)
            print("🎉 训练完成!")
            print("="*50)
            print(f"\n最佳模型: {best_model_path}")
            print(f"最佳F1微平均: {best_score:.4f}")
            
            # Save evaluation results
            eval_results = {
                'model_type': 'dual_encoder',
                'model_name': MODEL_NAME,
                'train_samples': len(dataset_train),
                'dev_samples': len(dataset_val),
                'best_val_f1_micro': best_score,
                's3_model_path': best_model_path,
                'epochs': EPOCHS,
                'batch_size': BATCH_SIZE,
                'learning_rate': LEARNING_RATE,
                'loss_type': LOSS_TYPE
            }
            
            eval_df = pd.DataFrame([eval_results])
            temp_results = os.path.join(tempfile.mkdtemp(), "evaluation_results.csv")
            eval_df.to_csv(temp_results, index=False)
            
            s3_key = f"{S3_PREFIX}/dual_encoder/evaluation_results.csv"
            success, s3_uri = upload_to_s3(temp_results, S3_BUCKET, s3_key)
            
            if success:
                print(f"\n评估结果已保存: {s3_uri}")
                os.remove(temp_results)
        else:
            print("\n⚠️ 未找到最佳模型")
    
    except Exception as e:
        print(f"\n❌ 训练失败: {str(e)}")
        import traceback
        traceback.print_exc()
    
    finally:
        # Cleanup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        del model
        import gc
        gc.collect()
        
        try:
            import shutil
            shutil.rmtree(temp_log_dir)
        except:
            pass
    
    print("\n✅ 训练流程结束")

print("✅ 训练Functions defined")

## 9. 执行训练

In [ ]:
# Start training
train_dual_encoder_model()

## 10. 预测函数（可选）

In [ ]:
def predict_with_dual_encoder(texts, explanations=None, s3_model_uri=None, threshold=0.5):
    """
    使用训练好的双编码器模型进行预测
    
    Args:
        texts: 文本列表
        explanations: 解释列表（可选）
        s3_model_uri: 模型在S3的URI
        threshold: 预测阈值
    
    Returns:
        list: 预测结果
    """
    # Download model
    local_model_path = os.path.join(tempfile.mkdtemp(), "model.ckpt")
    s3_parts = s3_model_uri.replace(ENDPOINT, '').strip('/').split('/', 1)
    bucket_name = s3_parts[0]
    s3_key = s3_parts[1]
    
    success = download_from_s3(bucket_name, s3_key, local_model_path)
    if not success:
        raise Exception(f"无法下载模型: {s3_model_uri}")
    
    # Load model
    model = DualEncoderClassifier.load_from_checkpoint(local_model_path)
    model.eval()
    model = model.to(device)
    
    # Load labels和分词器
    labels_name = load_label_classes()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # Encode texts
    text_inputs = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(device)
    
    # Encode explanations
    if explanations:
        exp_texts = [exp if exp and exp.strip() else "无解释" for exp in explanations]
        exp_inputs = tokenizer(
            exp_texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_EXPLANATION_LENGTH,
            return_tensors="pt"
        ).to(device)
    else:
        exp_inputs = None
    
    # Run inference
    with torch.no_grad():
        if exp_inputs:
            logits = model(
                text_inputs["input_ids"],
                text_inputs["attention_mask"],
                exp_inputs["input_ids"],
                exp_inputs["attention_mask"]
            )
        else:
            logits = model(
                text_inputs["input_ids"],
                text_inputs["attention_mask"]
            )
        probs = torch.sigmoid(logits)
    
    # Format results
    probs_numpy = probs.cpu().numpy()
    pred_labels = (probs_numpy > threshold).astype(int)
    
    results = []
    for i, text in enumerate(texts):
        result = {
            "text": text,
            "explanation": explanations[i] if explanations and i < len(explanations) else None,
            "probabilities": {labels_name[j]: float(probs_numpy[i, j]) for j in range(len(labels_name))},
            "predicted_labels": [labels_name[j] for j in range(len(labels_name)) if pred_labels[i, j] == 1]
        }
        results.append(result)
    
    # Cleanup
    os.remove(local_model_path)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return results

print("✅ 预测Functions defined")

## 11. 示例：使用模型预测

In [ ]:
# 示例代码（需要先训练模型）
# test_texts = [
#     "这是一个测试文本...",
#     "另一个测试文本..."
# ]
# test_explanations = [
#     "这个文本使用了...",
#     "这个文本包含..."
# ]
# 
# # 使用训练好的模型进行预测
# s3_model_uri = "你的模型S3路径"
# results = predict_with_dual_encoder(
#     texts=test_texts,
#     explanations=test_explanations,
#     s3_model_uri=s3_model_uri,
#     threshold=0.5
# )
# 
# for result in results:
#     print(f"文本: {result['text'][:50]}...")
#     print(f"预测标签: {result['predicted_labels']}")
#     print()